# **MultiFileChat - Chat with PDFs, Office files, spreadsheets, code and databases**

**Supported:** `.pdf` `.docx` `.pptx` `.csv` `.xlsx` `.sqlite/.sqlite3/.db` `.txt` `.md` `.json` `.yaml` `.html` `.xml`, and source code such as `.py` `.java` `.php` `.sql` `.js` `.c` `.cpp` `.cs` `.go` and more.

**How to use:** put your files in a folder named `files/` next to this notebook, then run the cells top to bottom.

**Not supported:** old `.doc` / `.xls` / `.ppt` (re-save as the newer format), images and scanned PDFs (no OCR), files with no extension.

## **Step-1:- Install required libraries**

In [1]:
%pip install -q PyPDF2 langchain langchain-google-genai google-generativeai faiss-cpu langchain-text-splitters langchain-community pandas openpyxl python-docx python-pptx

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## **Step-2:- Import Libraries**

In [2]:
import os
import glob
import getpass

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import google.generativeai as genai

C:\Users\panda\AppData\Local\Temp\ipykernel_31572\2738346463.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
c:\Users\panda\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\panda\AppData\Local\Temp\ipykernel_31572\2738346463.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob

## **Step-3:- Set Gemini API Key (securely)**

This reads the `GOOGLE_API_KEY` environment variable, or asks you to type it in (hidden input, not saved in the notebook).

In [3]:
api_key = os.environ.get("GOOGLE_API_KEY")

if not api_key:
    api_key = getpass.getpass("Enter your Gemini API key: ")
    os.environ["GOOGLE_API_KEY"] = api_key

genai.configure(api_key=api_key)
print("Gemini API key configured.")

Gemini API key configured.


## **Step-4:- File readers (one per file type -> plain text)**

To support another plain-text type, just add its extension to `TEXT_EXTENSIONS`. Only the first `MAX_ROWS` rows of each sheet/table are indexed, to keep embedding time and API quota reasonable.

In [4]:
import io
import os
import sqlite3
import tempfile

import pandas as pd
from PyPDF2 import PdfReader

# Tabular files / databases can be huge. Embedding every row is slow and costs
# API quota, so only the first MAX_ROWS rows of each sheet/table are indexed.
MAX_ROWS = 2000

# Anything that is already plain text (docs, config, source code, scripts).
TEXT_EXTENSIONS = {
    # DOCUMENTS / MARKUP
    "txt", "md", "markdown", "rst", "log", "html", "htm", "xml",
    # DATA / CONFIG
    "json", "yaml", "yml", "toml", "ini", "cfg", "env",
    # PROGRAMMING LANGUAGES
    "py", "java", "php", "js", "ts", "jsx", "tsx", "c", "cpp", "h", "hpp",
    "cs", "go", "rb", "rs", "kt", "swift", "scala", "lua", "pl", "r",
    # SCRIPTS / QUERIES
    "sql", "sh", "bat", "ps1", "css",
}
SQLITE_EXTENSIONS = {"sqlite", "sqlite3", "db"}
SUPPORTED_EXTENSIONS = sorted(
    {"pdf", "csv", "xlsx", "docx", "pptx"} | TEXT_EXTENSIONS | SQLITE_EXTENSIONS
)

def _decode(data: bytes) -> str:
    """Decode bytes as text, trying common encodings before giving up."""
    for encoding in ("utf-8-sig", "utf-16", "cp1252", "latin-1"):
        try:
            return data.decode(encoding)
        except (UnicodeDecodeError, UnicodeError):
            continue
    return data.decode("utf-8", errors="replace")

def _dataframe_to_text(df: pd.DataFrame, label: str, notes: list) -> str:
    """
    Turn a table into 'column: value' lines, one per row. Embedding models
    understand "Name: Asha; City: Pune" far better than a bare "Asha,Pune".
    """
    total_rows = len(df)
    if total_rows > MAX_ROWS:
        notes.append(f"{label}: only the first {MAX_ROWS:,} of {total_rows:,} rows were indexed.")
        df = df.head(MAX_ROWS)

    lines = [f"Table: {label}", f"Columns: {', '.join(map(str, df.columns))}"]
    for _, row in df.iterrows():
        cells = [f"{col}: {val}" for col, val in row.items() if pd.notna(val)]
        if cells:
            lines.append("; ".join(cells))
    return "\n".join(lines)


# INDIVIDUAL LOADERS
def _load_pdf(data, name, notes):
    reader = PdfReader(io.BytesIO(data))
    pages = []
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            pages.append(extracted)
    return "\n".join(pages)

def _load_text(data, name, notes):
    return _decode(data)

def _load_csv(data, name, notes):
    df = pd.read_csv(io.StringIO(_decode(data)), sep=None, engine="python")
    return _dataframe_to_text(df, name, notes)

def _load_xlsx(data, name, notes):
    sheets = pd.read_excel(io.BytesIO(data), sheet_name=None)  # needs openpyxl
    parts = [_dataframe_to_text(df, f"{name} / sheet '{sheet}'", notes)
             for sheet, df in sheets.items()]
    return "\n\n".join(parts)

def _load_docx(data, name, notes):
    from docx import Document  # python-docx
    doc = Document(io.BytesIO(data))
    parts = [p.text for p in doc.paragraphs if p.text.strip()]
    for table in doc.tables:
        for row in table.rows:
            cells = [cell.text.strip() for cell in row.cells]
            if any(cells):
                parts.append(" | ".join(cells))
    return "\n".join(parts)

def _load_pptx(data, name, notes):
    from pptx import Presentation  # python-pptx
    prs = Presentation(io.BytesIO(data))
    parts = []
    for number, slide in enumerate(prs.slides, start=1):
        parts.append(f"--- Slide {number} ---")
        for shape in slide.shapes:
            if shape.has_text_frame and shape.text_frame.text.strip():
                parts.append(shape.text_frame.text)
            if getattr(shape, "has_table", False) and shape.has_table:
                for row in shape.table.rows:
                    parts.append(" | ".join(cell.text.strip() for cell in row.cells))
        if slide.has_notes_slide and slide.notes_slide.notes_text_frame.text.strip():
            parts.append("Speaker notes: " + slide.notes_slide.notes_text_frame.text)
    return "\n".join(parts)

def _load_sqlite(data, name, notes):
    # sqlite3 needs a real file on disk, so write the upload to a temp file.
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".sqlite")
    try:
        tmp.write(data)
        tmp.close()
        conn = sqlite3.connect(tmp.name)
        try:
            tables = conn.execute(
                "SELECT name, sql FROM sqlite_master "
                "WHERE type='table' AND name NOT LIKE 'sqlite_%'"
            ).fetchall()
            parts = []
            for table_name, create_sql in tables:
                parts.append(f"Database table: {table_name}\nSchema: {create_sql}")
                df = pd.read_sql_query(f'SELECT * FROM "{table_name}"', conn)
                parts.append(_dataframe_to_text(df, f"{name} / table '{table_name}'", notes))
            return "\n\n".join(parts)
        finally:
            conn.close()
    finally:
        os.remove(tmp.name)


# PUBLIC ENTRY POINT
def load_file(name: str, data: bytes):
    """
    Returns (text, notes). `notes` is a list of human-readable warnings
    (e.g. truncated tables) the UI can show to the user.
    Raises ValueError for unsupported types.
    """
    ext = name.rsplit(".", 1)[-1].lower() if "." in name else ""
    notes = []
    if ext == "pdf":
        text = _load_pdf(data, name, notes)
    elif ext == "csv":
        text = _load_csv(data, name, notes)
    elif ext == "xlsx":
        text = _load_xlsx(data, name, notes)
    elif ext == "docx":
        text = _load_docx(data, name, notes)
    elif ext == "pptx":
        text = _load_pptx(data, name, notes)
    elif ext in SQLITE_EXTENSIONS:
        text = _load_sqlite(data, name, notes)
    elif ext in TEXT_EXTENSIONS:
        text = _load_text(data, name, notes)
    else:
        raise ValueError(f"Unsupported file type: .{ext}")
    return text, notes

## **Step-5:- Read files, split into chunks (tagged with their source file)**

In [5]:
def build_chunks(file_paths):
    splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1000)
    texts, metadatas, failed = [], [], []
    for path in file_paths:
        name = os.path.basename(path)
        try:
            with open(path, "rb") as fh:
                text, notes = load_file(name, fh.read())
        except Exception as e:  # one bad file shouldn't stop the rest
            failed.append(f"{name}: {e}")
            continue
        for note in notes:
            print("Note:", note)
        if not text.strip():
            failed.append(f"{name}: no readable text found")
            continue
        for chunk in splitter.split_text(text):
            texts.append(f"[Source: {name}]\n{chunk}")   # LLM also sees the source
            metadatas.append({"source": name})
    return texts, metadatas, failed

## **Step-6:- Create Embeddings & Store in FAISS (local folder)**

In [6]:
# Newer langchain-google-genai versions need the full "models/..." name.
# If your key can't access it, try "models/embedding-001" instead.
EMBEDDING_MODEL = "models/gemini-embedding-001"

def get_vector_store(texts, metadatas, index_path="faiss_index"):
    embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)
    vector_store = FAISS.from_texts(texts, embedding=embeddings, metadatas=metadatas)
    vector_store.save_local(index_path)
    print(f"FAISS vector store saved to ./{index_path}")

## **Step-7:- Create Conversational Chain (Q&A logic)**

In [7]:
def get_conversational_chain():
    prompt = ChatPromptTemplate.from_template("""
    Answer the question as detailed as possible using ONLY
    the information provided in the context.

    If the answer is not available in the context, say:
    "Answer is not available in the context."

    Do not use outside knowledge
    Do not make up information.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """)
    model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)
    return prompt | model | StrOutputParser()

## **Step-8:- Handle User Questions**

In [8]:
def user_input(user_question, index_path="faiss_index"):
    embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

    db = FAISS.load_local(index_path, embeddings, allow_dangerous_deserialization=True)
    docs = db.similarity_search(user_question, k=4)

    context = "\n\n".join(doc.page_content for doc in docs)
    sources = sorted({doc.metadata.get("source", "unknown") for doc in docs})

    chain = get_conversational_chain()
    res = chain.invoke({"context": context, "question": user_question})

    print("\nReply:")
    print(res)
    print("\nSources searched:", ", ".join(sources))
    return res

## **Step-9:- Main Orchestration Function**

Put your files in a folder named `files/`. Set `REBUILD_INDEX = True` whenever you add or change files, otherwise the saved index from the last run is reused.

In [10]:
REBUILD_INDEX = True   # set to False to reuse an existing ./faiss_index

def main():
    print("Chat with your files using Gemini")
    files_folder = "files"
    index_path = "faiss_index"
    if REBUILD_INDEX or not os.path.exists(index_path):
        file_paths = sorted(
            p for p in glob.glob(os.path.join(files_folder, "*"))
            if os.path.isfile(p) and p.rsplit(".", 1)[-1].lower() in SUPPORTED_EXTENSIONS
        )
        if not file_paths:
            print(f"No supported files found in ./{files_folder}/ - add some and re-run.")
            return
        print(f"\nProcessing {len(file_paths)} file(s): {[os.path.basename(p) for p in file_paths]}")
        texts, metadatas, failed = build_chunks(file_paths)
        for message in failed:
            print("Skipped", message)
        if not texts:
            print("No usable text found in the files.")
            return
        print(f"Created {len(texts)} chunks")
        get_vector_store(texts, metadatas, index_path)
    else:
        print("Using existing FAISS index.")
    while True:
        user_question = input("\nAsk a question about your files (or type 'exit' to quit): ")
        if user_question.lower().strip() == "exit":
            print("Goodbye!")
            break
        if not user_question.strip():
            print("Please enter a question.")
            continue
        user_input(user_question, index_path)

main()

Chat with your files using Gemini

Processing 7 file(s): ['Aakashvani-ShorlFilm.docx', 'Attention- is-all-You-need-research-paper.pdf', 'CLG-Birthdays.xlsx', 'GANGSTER-MOVIE.docx', 'OpenCart-TestPlan.docx', 'OrangeHRM-TestPlan.docx', 'YOLO-research-paper.pdf']
Created 14 chunks
FAISS vector store saved to ./faiss_index

Reply:
Sai ram was born on 2025-06-20.

Sources searched: Attention- is-all-You-need-research-paper.pdf, CLG-Birthdays.xlsx, OpenCart-TestPlan.docx, OrangeHRM-TestPlan.docx

Reply:
The OrangeHRM Test Plan defines the scope, testing approach, schedule, resources, deliverables, and quality criteria for validating the OrangeHRM Personal Information Management (PIM) module.

Here's a detailed explanation of the OrangeHRM Test Plan:

**1. Objective:**
The main objective is to validate the OrangeHRM PIM module. The testing aims to:
*   Verify that all employee information management functions work correctly.
*   Ensure data integrity and usability across all PIM submodules.
*

KeyboardInterrupt: Interrupted by user